# genesiss-20b — finetune on Text-to-CadQuery (Colab)

**gpt-oss-20b** is loaded in 4-bit (≈14 GB VRAM) — unsloth handles the MXFP4 MoE quantization.

Runs on Colab. Recommended accelerator: A100 / H100.

Pattern
1. Install deps + log in to HF.
2. Pull the latest checkpoint from `HUB_CKPT_REPO` if one exists.
3. Train with SFTTrainer — every save fires a callback that async-uploads the new checkpoint.
4. On finish, push merged 16-bit + GGUF Q4_K_M to `HUB_FINAL_REPO`.


## 1 · Install

In [ ]:
# Install Unsloth and friends. Colab — quiet install.
%%capture
!pip install --upgrade --quiet pip
!pip install --upgrade --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --quiet --no-deps "trl<0.10" peft accelerate bitsandbytes
!pip install --upgrade --quiet "huggingface_hub>=0.25" datasets tomli_w


## 2 · Authenticate with Hugging Face

In [ ]:
# Pull HF_TOKEN from Colab userdata.  Settings → Secrets → add `HF_TOKEN`.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    # Fallback: paste your token here if you really must.
    os.environ.setdefault("HF_TOKEN", "")
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)


## 3 · Vendor the shared helpers

In [ ]:
# Vendor training/shared/* into the runtime so we can `from shared import ...`.
# If you've cloned the repo into /content, you can skip this cell and just
# `import sys; sys.path.insert(0, "/content/genesiss/training")`.
import os, urllib.request, sys
SHARED_BASE = "https://raw.githubusercontent.com/numinousmuses/genesiss/main/training/shared"
TARGET = "/content/shared"
os.makedirs(TARGET, exist_ok=True)
for fname in ["__init__.py", "data_loader.py", "format.py", "checkpoint.py"]:
    url = f"{SHARED_BASE}/{fname}"
    try:
        urllib.request.urlretrieve(url, f"{TARGET}/{fname}")
    except Exception as e:
        print(f"[warn] could not fetch {fname} from GitHub ({e}). "
              "Paste the file contents manually if running before the repo is public.")
sys.path.insert(0, "/content")


## 4 · Run config

In [ ]:
# ---- run config ---------------------------------------------------------
VARIANT       = "genesiss-20b"
BASE_MODEL    = "unsloth/gpt-oss-20b"
FAMILY        = "gpt-oss"                     # qwen3.5 or gpt-oss

# Where checkpoints live on the Hub.  Each `checkpoint-XXXX` folder is a
# full Trainer save (model.safetensors, optimizer.pt, scheduler.pt,
# trainer_state.json, etc.) so we can resume mid-step.
HUB_CKPT_REPO = f"YOUR_HF_USER/{VARIANT}-checkpoints"
# Final merged model / GGUF goes here.
HUB_FINAL_REPO = f"YOUR_HF_USER/{VARIANT}"

MAX_SEQ_LENGTH               = 2048
PER_DEVICE_TRAIN_BATCH_SIZE  = 1
GRADIENT_ACCUMULATION_STEPS  = 4
LORA_R                       = 16
LORA_ALPHA                   = 16
LEARNING_RATE                = 0.0002
SAVE_STEPS                   = 100
NUM_EPOCHS                   = 1

# Local dir Trainer writes checkpoints to (then async-pushed to HUB_CKPT_REPO).
OUTPUT_DIR = f"./outputs/{VARIANT}"


## 5 · Resume from Hub (if any)

In [ ]:
# Resume from the Hub if a prior session uploaded a checkpoint.
# `pull_latest_checkpoint` returns the local path, or None.
from shared.checkpoint import pull_latest_checkpoint
resumed_path = pull_latest_checkpoint(HUB_CKPT_REPO, OUTPUT_DIR, token=os.environ["HF_TOKEN"])
print("resumed from:", resumed_path)


## 6 · Load base model + attach LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
            max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    max_seq_length = MAX_SEQ_LENGTH,
)


## 7 · Load dataset + apply chat template

In [ ]:
# Load Text-to-CadQuery (ricemonster/NeurIPS11092) and convert to a `text` column
# ready for SFTTrainer.  Set max_train=None for full 90% split (~150k rows).
from shared.data_loader import load_dataset_dict
from shared.format import text_field

ds = load_dataset_dict(max_train=None, max_eval=2000)
ds = ds.map(lambda b: {
    "text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in b["messages"]
    ]
}, batched=True, remove_columns=["messages"])

print(ds)
print("--- example ---")
print(ds["train"][0]["text"][:1200])


## 8 · Build trainer (with async Hub-upload callback)

In [ ]:
from trl import SFTTrainer, SFTConfig
from shared.checkpoint import make_hub_callback

hub_cb = make_hub_callback(HUB_CKPT_REPO, OUTPUT_DIR, token=os.environ["HF_TOKEN"])

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds["validation"],
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        max_seq_length = MAX_SEQ_LENGTH,
        dataset_text_field = "text",
        per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        warmup_steps = 20,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 3407,
        # Save every SAVE_STEPS — the HubCheckpointCallback async-pushes
        # each checkpoint to HUB_CKPT_REPO so we can resume next session.
        save_strategy = "steps",
        save_steps = SAVE_STEPS,
        save_total_limit = 2,
        evaluation_strategy = "steps",
        eval_steps = SAVE_STEPS,
        report_to = "none",
        dataset_num_proc = 2,
    ),
    callbacks = [hub_cb],
)


## 9 · Train

In [ ]:
# `resume_from_checkpoint=True` makes the Trainer look in OUTPUT_DIR for the
# latest checkpoint-XXXX (the one we just pulled, if any).
trainer_stats = trainer.train(resume_from_checkpoint=True if resumed_path else False)
print(trainer_stats)


## 10 · Smoke test

In [ ]:
# Quick smoke test on the trained adapter — does it emit CadQuery?
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content":
        "You are Genesiss, a CAD assistant. Output ONLY CADQuery python."},
    {"role": "user", "content": "a 30mm cube with a 10mm hole through the centre"},
]
ids = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt"
).to(model.device)
out = model.generate(ids, max_new_tokens=400, do_sample=False)
print(tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True))


## 11 · Export (merged 16-bit + GGUF for Ollama)

In [ ]:
# ---- export -------------------------------------------------------------
# 1) Push merged 16-bit weights (for HF inference, vLLM, transformers).
model.push_to_hub_merged(
    HUB_FINAL_REPO,
    tokenizer,
    save_method = "merged_16bit",
    token = os.environ["HF_TOKEN"],
)

# 2) GGUF Q4_K_M for Ollama.  Drop this next to genesiss-20b.Modelfile, then:
#    ollama create genesiss-20b -f genesiss-20b.Modelfile
model.save_pretrained_gguf(
    f"./gguf-{VARIANT}",
    tokenizer,
    quantization_method = "q4_k_m",
)

# 3) Also push the GGUF to the Hub so the CLI's `genesiss models pull` can find it.
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.upload_folder(
    folder_path = f"./gguf-{VARIANT}",
    repo_id = HUB_FINAL_REPO,
    repo_type = "model",
    path_in_repo = "gguf",
    commit_message = "merged GGUF Q4_K_M",
)
